In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 00_common
# MAGIC Configuracao, helpers e DDL compartilhados. Nao precisa rodar sozinho: os outros notebooks chamam com `%run ./00_common`.
# MAGIC
# MAGIC O notebook no Databricks **precisa se chamar `00_common`** e ficar na mesma pasta dos demais.

# COMMAND ----------

import json
import time
import uuid
from datetime import datetime, timedelta, timezone

import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, ArrayType
)

# Todos os timestamps do projeto sao UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "workspace"
SCHEMA = "github_incremental"
FQ = f"`{CATALOG}`.`{SCHEMA}`"

BRONZE = f"{FQ}.bronze_issues"
SILVER = f"{FQ}.silver_issues"
WATERMARKS = f"{FQ}.etl_watermarks"
RUN_LOG = f"{FQ}.etl_run_log"
QUALITY = f"{FQ}.etl_quality_results"
GOLD_REPO_ACTIVITY = f"{FQ}.gold_repo_activity"
GOLD_DAILY_ACTIVITY = f"{FQ}.gold_daily_activity"

TS_FMT = "yyyy-MM-dd'T'HH:mm:ssX"

# COMMAND ----------

# Helpers de tempo. Tudo trafega como string "yyyy-MM-dd HH:mm:ss[.ffffff]" em UTC
# e so vira TIMESTAMP dentro do Spark (evita ambiguidade de fuso no driver Python).

def now_str():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")


def new_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6]


def to_iso(ts_str):
    """'2026-06-16 14:30:00.123' -> '2026-06-16T14:30:00Z' (formato aceito pelo parametro since do GitHub)."""
    return ts_str[:19].replace(" ", "T") + "Z"


def from_iso(iso):
    """'2026-06-16T14:30:00Z' -> '2026-06-16 14:30:00'."""
    return iso.replace("T", " ").rstrip("Z")

# COMMAND ----------

# Watermarks: uma linha por entidade ("extract:<owner/repo>" e "silver_issues").
# coverage_start = a partir de quando os dados daquela entidade estao completos.

def get_watermark(entity):
    row = (
        spark.table(WATERMARKS)
        .filter(F.col("entity") == entity)
        .select(
            F.col("watermark").cast("string").alias("wm"),
            F.col("coverage_start").cast("string").alias("cov"),
        )
        .first()
    )
    return (row["wm"], row["cov"]) if row else (None, None)


def set_watermark(entity, wm_str, run_id, coverage_str=None):
    """Upsert do watermark. Nunca anda para tras (greatest) e coverage_start so diminui (least)."""
    cov = f"TIMESTAMP'{coverage_str}'" if coverage_str else "CAST(NULL AS TIMESTAMP)"
    spark.sql(f"""
        MERGE INTO {WATERMARKS} AS t
        USING (
            SELECT '{entity}' AS entity,
                   TIMESTAMP'{wm_str}' AS watermark,
                   {cov} AS coverage_start,
                   '{run_id}' AS last_run_id,
                   current_timestamp() AS updated_at
        ) AS s
        ON t.entity = s.entity
        WHEN MATCHED THEN UPDATE SET
            watermark = greatest(t.watermark, s.watermark),
            coverage_start = CASE
                WHEN s.coverage_start IS NULL THEN t.coverage_start
                WHEN t.coverage_start IS NULL THEN s.coverage_start
                ELSE least(t.coverage_start, s.coverage_start) END,
            last_run_id = s.last_run_id,
            updated_at = s.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)

# COMMAND ----------

# Log de execucao (uma linha por etapa/entidade).

RUN_LOG_INPUT_SCHEMA = StructType([
    StructField("run_id", StringType()),
    StructField("step", StringType()),
    StructField("entity", StringType()),
    StructField("started_at", StringType()),
    StructField("finished_at", StringType()),
    StructField("status", StringType()),
    StructField("rows_in", LongType()),
    StructField("rows_inserted", LongType()),
    StructField("rows_updated", LongType()),
    StructField("detail", StringType()),
])


def log_run(run_id, step, entity, started_at, status,
            rows_in=None, rows_inserted=None, rows_updated=None, detail=None):
    row = [(run_id, step, entity, started_at, now_str(), status,
            rows_in, rows_inserted, rows_updated, (detail or "")[:2000])]
    (
        spark.createDataFrame(row, RUN_LOG_INPUT_SCHEMA)
        .withColumn("started_at", F.col("started_at").cast("timestamp"))
        .withColumn("finished_at", F.col("finished_at").cast("timestamp"))
        .write.format("delta").mode("append").saveAsTable(RUN_LOG)
    )

# COMMAND ----------

# Parse do JSON cru (bronze) -> colunas tipadas, e deduplicacao dentro do lote.

PAYLOAD_SCHEMA = StructType([
    StructField("number", LongType()),
    StructField("title", StringType()),
    StructField("state", StringType()),
    StructField("comments", LongType()),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("closed_at", StringType()),
    StructField("user", StructType([StructField("login", StringType())])),
    StructField("labels", ArrayType(StructType([StructField("name", StringType())]))),
    StructField("pull_request", StructType([StructField("url", StringType())])),
])


def build_silver_batch(bronze_df):
    """Uma linha por (repo, issue_id): a versao com maior updated_at do lote."""
    parsed = bronze_df.withColumn("j", F.from_json("payload", PAYLOAD_SCHEMA))
    flat = parsed.select(
        "repo",
        "issue_id",
        F.col("j.number").alias("number"),
        F.col("j.title").alias("title"),
        F.col("j.state").alias("state"),
        F.col("j.pull_request").isNotNull().alias("is_pull_request"),
        F.col("j.user.login").alias("author_login"),
        F.expr("transform(j.labels, x -> x.name)").alias("labels"),
        F.col("j.comments").alias("comments"),
        F.to_timestamp(F.col("j.created_at"), TS_FMT).alias("created_at"),
        F.to_timestamp(F.col("j.updated_at"), TS_FMT).alias("updated_at"),
        F.to_timestamp(F.col("j.closed_at"), TS_FMT).alias("closed_at"),
        "run_id",
        "ingested_at",
    )
    w = Window.partitionBy("repo", "issue_id").orderBy(
        F.col("updated_at").desc(), F.col("ingested_at").desc()
    )
    return (
        flat.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
        .withColumn("_first_seen_at", F.current_timestamp())
        .withColumn("_merged_at", F.current_timestamp())
    )


def merge_silver(batch_df, view_name="silver_batch"):
    """MERGE condicional: so atualiza se a versao do lote for mais nova que a do Silver."""
    batch_df.createOrReplaceTempView(view_name)
    skip_on_update = {"repo", "issue_id", "_first_seen_at"}
    set_clause = ", ".join(
        f"`{c}` = s.`{c}`" for c in batch_df.columns if c not in skip_on_update
    )
    res = spark.sql(f"""
        MERGE INTO {SILVER} AS t
        USING {view_name} AS s
        ON t.repo = s.repo AND t.issue_id = s.issue_id
        WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    m = res.asDict() if res is not None else {}
    return {"inserted": m.get("num_inserted_rows"), "updated": m.get("num_updated_rows")}

# COMMAND ----------

# Criacao idempotente de schema e tabelas.

def ensure_objects():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {BRONZE} (
            repo STRING, issue_id BIGINT, updated_at STRING,
            payload STRING, run_id STRING, ingested_at TIMESTAMP
        ) USING DELTA
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER} (
            repo STRING, issue_id BIGINT, number BIGINT, title STRING, state STRING,
            is_pull_request BOOLEAN, author_login STRING, labels ARRAY<STRING>,
            comments BIGINT, created_at TIMESTAMP, updated_at TIMESTAMP, closed_at TIMESTAMP,
            run_id STRING, ingested_at TIMESTAMP,
            _first_seen_at TIMESTAMP, _merged_at TIMESTAMP
        ) USING DELTA
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {WATERMARKS} (
            entity STRING, watermark TIMESTAMP, coverage_start TIMESTAMP,
            last_run_id STRING, updated_at TIMESTAMP
        ) USING DELTA
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {RUN_LOG} (
            run_id STRING, step STRING, entity STRING,
            started_at TIMESTAMP, finished_at TIMESTAMP, status STRING,
            rows_in BIGINT, rows_inserted BIGINT, rows_updated BIGINT, detail STRING
        ) USING DELTA
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {QUALITY} (
            check_name STRING, layer STRING, expected STRING, observed STRING,
            status STRING, run_id STRING, checked_at TIMESTAMP
        ) USING DELTA
    """)

# COMMAND ----------

ensure_objects()
print(f"Objetos prontos em {CATALOG}.{SCHEMA}")